# Full-Packet Partial Stress Viewer

This notebook reads saved quick-analysis outputs only. It does not iterate over raw files.

The data is partial/debug stress-test output. It is not a complete week, not a complete first day, and not a validated all-alert census.

In [ ]:
from pathlib import Path
import json
import pandas as pd

OUT_DIR = Path('../outputs/data_transfer/quick_analysis/full_week_full_packet_partial').resolve()
OUT_DIR

In [ ]:
metadata_path = OUT_DIR / 'raw_metadata_summary.json'
sample_summary_path = OUT_DIR / 'sample_5000_summary.json'
file_rows_path = OUT_DIR / 'file_rows.parquet'
sample_path = OUT_DIR / 'sample_5000_files_flattened.parquet'

metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
sample_summary = json.loads(sample_summary_path.read_text()) if sample_summary_path.exists() else {}
metadata

## Raw Metadata Summary

In [ ]:
pd.DataFrame([{
    'files': metadata.get('file_count'),
    'readable_files': metadata.get('readable_file_count'),
    'readable_rows': metadata.get('readable_rows'),
    'expected_total': metadata.get('expected_total'),
    'apparent_percent_complete': metadata.get('apparent_percent_complete'),
    'schema_groups': metadata.get('schema_group_count'),
    'scientific_completeness': metadata.get('scientific_completeness'),
}])

## File Row Distribution

In [ ]:
file_rows = pd.read_parquet(file_rows_path) if file_rows_path.exists() else pd.DataFrame()
file_rows['rows'].describe() if 'rows' in file_rows else file_rows

## Flattened Sample Preview

In [ ]:
sample = pd.read_parquet(sample_path) if sample_path.exists() else pd.DataFrame()
sample.head()

## Band Counts And IDs

In [ ]:
pd.DataFrame({
    'band': list((sample_summary.get('band_counts') or {}).keys()),
    'rows': list((sample_summary.get('band_counts') or {}).values()),
})

In [ ]:
{
    'unique_objects': sample_summary.get('unique_objects'),
    'unique_sources': sample_summary.get('unique_sources'),
    'object_column': sample_summary.get('object_column'),
    'source_column': sample_summary.get('source_column'),
}

## Classifier, Time, Sky, And Missingness

In [ ]:
sample_summary.get('classifier_columns', [])

In [ ]:
{
    'time_coverage': sample_summary.get('time_coverage'),
    'sky_coverage': sample_summary.get('sky_coverage'),
}

In [ ]:
missingness = sample_summary.get('missingness') or {}
pd.Series(missingness).sort_values(ascending=False).head(30)

## Limitations

- Partial debug/stress-test only.
- Not a complete week.
- Not a complete first day.
- Not a validated all-alert census.
- Useful for schema, performance, and data-quality analysis only.